# Generating relationship perturbations (edge perturbation)

All previous deficits change the *values* of columns. **Edge
perturbation** changes something else: the **strength of the
relationship between two columns**. The correlation between X and Y
becomes stronger (or flips), while the marginal distributions stay
essentially the same. Use this to test pipelines that assume a fixed
correlation structure — feature selection, regularized regressions,
multicollinearity handling.

This deficit only exists on the **data-generating process (DGP)**
side: `CausalEdgePerturbationGenerator` scales one edge coefficient of
a *fitted causal model* and recomputes the child (and its
descendants), so the change is structurally consistent, not a
hand-crafted covariance trick.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx

from badgers.core.dgp import OlsDGP
from badgers.generators.tabular_data.relationship.causal import CausalEdgePerturbationGenerator


In [ ]:
def generate_linear_data(graph, n_samples=100, rng=None, coefficients=None):
    """
    Generate data from a linear structural equation model on a DAG.
    Each edge (u -> v) carries a coefficient w. Every node value is
        v = sum_{u in parents(v)} w_{u,v} * u + noise
    with noise ~ N(0, 1).
    """
    if rng is None:
        rng = np.random.default_rng(42)
    if coefficients is None:
        coefficients = {edge: 1.0 for edge in graph.edges()}

    nodes = list(graph.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    X = np.zeros((n_samples, len(nodes)))
    for node in nx.topological_sort(graph):
        idx = node_to_idx[node]
        parents = list(graph.predecessors(node))
        if not parents:
            X[:, idx] = rng.normal(size=n_samples)
        else:
            X[:, idx] = rng.normal(size=n_samples)
            for parent in parents:
                coef = coefficients.get((parent, node), 1.0)
                X[:, idx] += coef * X[:, node_to_idx[parent]]
    return X, {node: i for i, node in enumerate(nodes)}


def draw_graph(graph, title=None):
    """
    Draw a DAG with node labels.
    """
    fig, ax = plt.subplots(1, figsize=(6, 2))
    pos = nx.spring_layout(graph, seed=42)
    nx.draw(graph, pos, ax=ax, node_color="lightblue", node_size=800, arrowsize=15)
    nx.draw_networkx_labels(graph, pos, ax=ax)
    if title:
        ax.set_title(title, fontsize=12)
    ax.axis("off")
    return fig


def plot_2panel(X, Xt, x_idx=0, y_idx=1, x_name="X", y_name="Y"):
    """
    Original | Transformed scatter of two dimensions.
    """
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
    sns.scatterplot(x=X[:, x_idx], y=X[:, y_idx], color="C0", ax=axes[0])
    sns.scatterplot(x=Xt[:, x_idx], y=Xt[:, y_idx], color="C1", ax=axes[1])
    axes[0].set_title("Original")
    axes[1].set_title("Transformed")
    axes[0].set_xlabel(x_name)
    axes[1].set_xlabel(x_name)
    axes[0].set_ylabel(y_name)
    axes[1].set_ylabel(y_name)
    plt.tight_layout()
    return fig


## Build the causal graph and generate data

The causal generators take a **causal graph** (a DAG) as input. The
graph plays two roles:

1. **It defines the data.** We generate the data *from* the graph using
   a linear structural equation model, so the graph genuinely explains
   how the columns are related.
2. **It drives the perturbation.** When an edge is perturbed, the
   effect propagates to the descendants of the child along the
   directed edges and nowhere else.

We use a simple **chain** `X → Y → Z` with a deliberately *weak*
`X → Y` edge (coefficient 0.3) — the reason becomes clear below.

In [ ]:
chain = nx.DiGraph()
chain.add_edges_from([("X", "Y"), ("Y", "Z")])

draw_graph(chain, title="Chain: X → Y → Z")
plt.show()

In [ ]:
rng = np.random.default_rng(42)

X, column_mapping = generate_linear_data(
    chain, n_samples=100, rng=rng,
    coefficients={("X", "Y"): 0.3, ("Y", "Z"): -1},
)
print(f"X shape: {X.shape}")
print(f"column_mapping: {column_mapping}")

In [ ]:
sns.pairplot(pd.DataFrame(X, columns=column_mapping.keys()))

## Perturb an edge

`CausalEdgePerturbationGenerator` changes the **strength of a single
edge** and recomputes the child with the new coefficient — its
structural equation itself changed, and the child's **marginal
variance is preserved** by adjusting its residual noise. The child's
descendants are then recomputed via Pearl surgery with fresh noise:
each descendant is re-evaluated from its (shifted) parents through the
fitted structural equation, so descendants move because their parents
moved — not because a rigid delta was added on top.

We double the `X → Y` coefficient (`weight_factor=2.0`): the X → Y
relationship becomes stronger while Y's overall spread stays the
same.

Two constraints matter:

- `perturbation_nodes` must be the **child** of the perturbed edge —
  here `["Y"]`.
- The perturbation is only **feasible** while
  `beta'^2 * var(parent) <= var(child)`. That is why we used a *weak*
  `X → Y` edge (coefficient 0.3): with a strong edge, doubling it
  would push the child's implied variance above its observed variance
  and the generator would raise `ValueError`.


In [ ]:
dgp = OlsDGP(chain, column_mapping).fit(X.copy())

gen = CausalEdgePerturbationGenerator(random_generator=np.random.default_rng(0), dgp=dgp)


In [ ]:
Xt, _ = gen.generate(
    X.copy(), None,
    perturbation_nodes=["Y"],
    edge=("X", "Y"),
    weight_factor=2.0,
)
print(f"output shape: {Xt.shape}")

In [ ]:
Xt[:5]

### The transformed X → Y relationship

In the transformed panel the points hug the (steeper) X → Y line more
tightly — the correlation is stronger — while Y's marginal spread is
essentially unchanged:

In [ ]:
_ = plot_2panel(X, Xt, x_name="X", y_name="Y")

### Propagation to Z

Z is recomputed from its (shifted) parent through the fitted Y → Z
equation with fresh noise: since Y moved, Z's mean moves with it via
the fitted `beta_YZ`, while Z's own residual noise is freshly drawn.

In [ ]:
_ = plot_2panel(X, Xt, x_idx=1, y_idx=2, x_name="Y", y_name="Z")

### Infeasible perturbation

Pushing `weight_factor` too far violates the feasibility bound. On
this weak edge, a factor of 10 is infeasible:

In [ ]:
try:
    gen.generate(X.copy(), None,
                 perturbation_nodes=["Y"],
                 edge=("X", "Y"), weight_factor=10.0)
except ValueError as e:
    print(f"ValueError: {e}")

## Summary

| Lever | Generator | Uses a DGP? | Propagates? | Effect on shape ||---|---|---|---|---|
| DGP (edge coefficient) | `CausalEdgePerturbationGenerator` | **yes** (fitted) | **yes** — child + descendants re-evaluated | same shape (gating, no row change) |
* Edge perturbation scales one coefficient and preserves the child's
  marginal variance.
* `perturbation_nodes` must be the **child** of the edge.
* Feasibility requires `beta'^2 * var(parent) <= var(child)` — use
  weaker edges for larger `weight_factor`.
